## 1. Setup & Imports

In [ ]:
import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Auto-detect Gemma model directory (handles Kaggle's nested layout) ────────
_MODEL_ROOT = Path('/kaggle/input/models/google/gemma-3/transformers/gemma-3-4b-it/1')

   # root added by Kaggle hub
_config_files = sorted(_MODEL_ROOT.rglob('config.json'))
if not _config_files:
    print("ERROR: No config.json found. Listing tree:")
    for p in sorted(_MODEL_ROOT.rglob('*'))[:30]:
        print(' ', p)
    raise FileNotFoundError("Attach the Gemma-3 model: Add Data → Models → gemma-3-4b-it")

MODEL_PATH = _config_files[0].parent        # ← the real weights directory
print(f"✅ Model resolved: {MODEL_PATH}")

# ── Label columns ──────────────────────────────────────────────────────────────
LABEL_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA',
    'Effusion', "Synovitis", "Baker's", 'Contusion', 'Fracture',
]

print('Data dir:', DATA_DIR)
print('Files:', [f.name for f in DATA_DIR.iterdir() if f.is_file()])


## 2. Load Training Data

In [ ]:
# Read train.csv — handles embedded newlines in Report field
train_df = pd.read_csv(
    DATA_DIR / 'train.csv',
    quoting=1,   # QUOTE_ALL
    engine='python',
    on_bad_lines='skip',
)
print(f'Train studies: {len(train_df)}')
print(train_df.head(2))

# Separate labeled vs. report-only studies
has_labels = train_df[LABEL_COLS].notna().any(axis=1)
labeled_df   = train_df[has_labels].copy()
unlabeled_df = train_df[~has_labels].copy()
print(f'\nStudies with gold labels: {len(labeled_df)}')
print(f'Studies with report only: {len(unlabeled_df)}')

## 3. Fast Keyword Extraction (Pass 1)

In [ ]:
# ── Keyword lists per label (English terms)
# The LLM will handle other languages; keywords give a quick first pass.
KEYWORDS_POS = {
    'ACL':              ['acl tear', 'acl rupture', 'anterior cruciate', 'acl injury', 'acl lesion', 'anterior cruciate ligament tear'],
    'MCL':              ['mcl tear', 'mcl injury', 'medial collateral', 'mcl rupture', 'medial collateral ligament tear'],
    'Medial Meniscus':  ['medial meniscus tear', 'medial meniscal tear', 'medial meniscus lesion', 'medial meniscus injury'],
    'Lateral Meniscus': ['lateral meniscus tear', 'lateral meniscal tear', 'lateral meniscus lesion'],
    'Medial OA':        ['medial osteoarthritis', 'medial compartment oa', 'medial tibiofemoral oa', 'medial joint space narrowing', 'medial chondral'],
    'Lateral OA':       ['lateral osteoarthritis', 'lateral compartment oa', 'lateral tibiofemoral oa', 'lateral joint space narrowing'],
    'PF OA':            ['patellofemoral osteoarthritis', 'pf oa', 'patellofemoral oa', 'patellofemoral chondromalacia'],
    'Effusion':         ['joint effusion', 'effusion', 'fluid', 'intra-articular fluid'],
    'Synovitis':        ['synovitis', 'synovial', 'synovial thickening'],
    "Baker's":          ["baker's cyst", 'popliteal cyst', 'baker cyst'],
    'Contusion':        ['bone contusion', 'bone bruise', 'contusion', 'trabecular edema', 'bone marrow edema'],
    'Fracture':         ['fracture', 'fract', 'avulsion'],
}

KEYWORDS_NEG = {
    'ACL':              ['acl intact', 'acl normal', 'no acl tear', 'acl unremarkable'],
    'MCL':              ['mcl intact', 'no mcl tear', 'mcl normal'],
    'Medial Meniscus':  ['medial meniscus intact', 'no medial meniscus tear'],
    'Lateral Meniscus': ['lateral meniscus intact', 'no lateral meniscus tear'],
    'Fracture':         ['no fracture', 'no fx', 'fracture excluded'],
    'Effusion':         ['no effusion', 'no joint fluid'],
    "Baker's":          ["no baker's cyst", 'no popliteal cyst'],
}

def keyword_extract(report: str) -> dict:
    """Return {label: 0/1/NaN} via keyword matching."""
    if not isinstance(report, str) or len(report.strip()) == 0:
        return {col: np.nan for col in LABEL_COLS}

    text = report.lower()
    result = {}

    for col in LABEL_COLS:
        # Check negative first (higher precision)
        neg_hits = [kw for kw in KEYWORDS_NEG.get(col, []) if kw in text]
        pos_hits = [kw for kw in KEYWORDS_POS.get(col, []) if kw in text]

        if neg_hits and not pos_hits:
            result[col] = 0
        elif pos_hits:
            result[col] = 1
        else:
            result[col] = np.nan  # uncertain → send to LLM

    return result


# Apply keyword extraction
tqdm.pandas(desc='Keyword extraction')
kw_rows = train_df['Report'].progress_apply(keyword_extract).tolist()
kw_df = pd.DataFrame(kw_rows, index=train_df['StudyInstanceUID'])

# Fraction confident
confident = kw_df.notna().mean()
print('\nFraction of studies with confident keyword labels per condition:')
print(confident.round(3).to_string())

## 4. LLM Structured Extraction (Pass 2)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path_str = str(MODEL_PATH.resolve())   # ← key fix

tokenizer = AutoTokenizer.from_pretrained(
    model_path_str,
    local_files_only=True,    # ← prevents HF hub lookup
)
llm = AutoModelForCausalLM.from_pretrained(
    model_path_str,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    local_files_only=True,    # ← prevents HF hub lookup
)
llm.eval()
print('LLM loaded successfully.')

In [ ]:
SYSTEM_PROMPT = """You are a radiologist assistant that reads knee MRI reports.
For each report, extract binary presence (1) or absence (0) of exactly 12 findings.
If a finding is not mentioned, output 0.
Respond ONLY with a valid JSON object with exactly these 12 keys:
ACL, MCL, "Medial Meniscus", "Lateral Meniscus",
"Medial OA", "Lateral OA", "PF OA",
Effusion, Synovitis, "Baker's", Contusion, Fracture
Each value must be 0 or 1. No explanations, no text outside the JSON."""

FEW_SHOT = [
    {
        "role": "user",
        "content": "Report: Complete tear of the ACL. No meniscal tear identified. Small joint effusion present. No fracture."
    },
    {
        "role": "assistant",
        "content": '{"ACL": 1, "MCL": 0, "Medial Meniscus": 0, "Lateral Meniscus": 0, "Medial OA": 0, "Lateral OA": 0, "PF OA": 0, "Effusion": 1, "Synovitis": 0, "Baker\'s": 0, "Contusion": 0, "Fracture": 0}'
    },
]

MAX_REPORT_TOKENS = 800

def llm_extract(report: str, max_new_tokens: int = 150) -> dict:
    """Run LLM inference on a single report, return parsed JSON dict."""
    # Truncate long reports
    tokens = tokenizer.encode(report, add_special_tokens=False)
    if len(tokens) > MAX_REPORT_TOKENS:
        tokens = tokens[:MAX_REPORT_TOKENS]
        report = tokenizer.decode(tokens, skip_special_tokens=True)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *FEW_SHOT,
        {"role": "user", "content": f"Report: {report}"},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Parse JSON
    try:
        # Find JSON in response
        json_match = re.search(r'\{[^}]+\}', response, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            # Validate and sanitize
            result = {}
            for col in LABEL_COLS:
                val = data.get(col, 0)
                result[col] = int(val) if val in (0, 1, '0', '1') else 0
            return result
    except Exception:
        pass

    # Fallback: return zeros
    return {col: 0 for col in LABEL_COLS}


print('LLM extraction function defined.')
print('\nTest on a sample report:')
sample_report = train_df['Report'].dropna().iloc[0]
print('Report snippet:', sample_report[:200])
sample_result = llm_extract(sample_report)
print('Extracted labels:', sample_result)

In [ ]:
# ── Determine which studies need LLM (those with any NaN from keyword pass)
# Run LLM on ALL non-gold studies (keyword pass used as initialization)

# Studies without gold labels
unlabeled_uids = unlabeled_df['StudyInstanceUID'].tolist()
print(f'Running LLM on {len(unlabeled_uids)} non-gold studies...')

llm_results = []
for uid in tqdm(unlabeled_uids, desc='LLM extraction'):
    row = train_df[train_df['StudyInstanceUID'] == uid]
    report = row['Report'].values[0] if len(row) > 0 else ''

    if not isinstance(report, str) or len(report.strip()) < 10:
        llm_results.append({col: 0 for col in LABEL_COLS})
        continue

    extracted = llm_extract(report)
    llm_results.append(extracted)

llm_df = pd.DataFrame(llm_results, index=unlabeled_uids)
print('LLM extraction complete.')
print('Label prevalence from LLM:')
print(llm_df.mean().round(3).to_string())

## 5. Validate Against Gold Labels

In [ ]:
# Validate LLM quality against gold-standard labeled studies
gold_uids = labeled_df['StudyInstanceUID'].tolist()
print(f'Validating on {len(gold_uids)} gold-labeled studies...')

gold_llm_results = []
for uid in tqdm(gold_uids, desc='LLM validation pass'):
    row = train_df[train_df['StudyInstanceUID'] == uid]
    report = row['Report'].values[0] if len(row) > 0 else ''
    extracted = llm_extract(report) if isinstance(report, str) else {col: 0 for col in LABEL_COLS}
    gold_llm_results.append(extracted)

gold_llm_df = pd.DataFrame(gold_llm_results, index=gold_uids)

# Compare to actual gold labels
gold_true = labeled_df.set_index('StudyInstanceUID')[LABEL_COLS]

print('\nPer-label AUC (LLM vs. Gold Labels):')
aucs = {}
for col in LABEL_COLS:
    t = gold_true[col].fillna(0).values
    p = gold_llm_df[col].values
    if len(np.unique(t)) > 1:
        auc = roc_auc_score(t, p)
        aucs[col] = auc
        print(f'  {col:25s}: {auc:.4f}')
    else:
        aucs[col] = np.nan
        print(f'  {col:25s}: N/A (no positives in gold set)')

valid_aucs = [v for v in aucs.values() if not np.isnan(v)]
print(f'\nMacro-average AUC (LLM pseudo-labeler quality): {np.mean(valid_aucs):.4f}')

## 6. Build & Save Final Pseudo-Label CSV

In [ ]:
# Merge: gold labels take priority, LLM labels for the rest
pseudo_rows = []

for _, row in train_df.iterrows():
    uid = row['StudyInstanceUID']
    entry = {'StudyInstanceUID': uid}

    if uid in gold_true.index:
        # Gold labels — use directly
        for col in LABEL_COLS:
            val = gold_true.loc[uid, col]
            entry[col] = float(val) if not pd.isna(val) else 0.0
        entry['label_source'] = 'gold'
    elif uid in llm_df.index:
        # LLM-derived pseudo-label
        for col in LABEL_COLS:
            entry[col] = float(llm_df.loc[uid, col])
        entry['label_source'] = 'llm'
    else:
        # No report or couldn't process → neutral
        for col in LABEL_COLS:
            entry[col] = 0.0
        entry['label_source'] = 'default'

    pseudo_rows.append(entry)

pseudo_df = pd.DataFrame(pseudo_rows)
pseudo_df.to_csv(OUTPUT_DIR / 'train_pseudolabels.csv', index=False)

print(f'Pseudo-label CSV saved: {OUTPUT_DIR / "train_pseudolabels.csv"}')
print(f'\nSource breakdown:')
print(pseudo_df['label_source'].value_counts().to_string())
print(f'\nLabel prevalence in pseudo-labels:')
print(pseudo_df[LABEL_COLS].mean().round(3).to_string())
print(f'\nShape: {pseudo_df.shape}')
pseudo_df.head()